# 03 — Baseline: Exact Normalized String Matching

This notebook runs the exact-normalized-string clustering baseline (`src/exact_matching.py`) within topic boundaries, evaluates it with pairwise Precision, Recall, and F1 (`src/evaluation.py`), and inspects representative failure modes.


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)
print("Project root:", ROOT)


Project root: E:\AOU\elevvo internship\tusk 4


In [2]:
from src.data_loader import load_config
from src.exact_matching import cluster_exact_matching
from src.evaluation import compute_pairwise_metrics

cfg = load_config(ROOT / "config.yaml")
proc_dir = ROOT / cfg["paths"]["processed_dir"]

val_mentions = pd.read_parquet(proc_dir / "validation_mentions.parquet")
test_mentions = pd.read_parquet(proc_dir / "test_mentions.parquet")

val_exact = cluster_exact_matching(val_mentions)
test_exact = cluster_exact_matching(test_mentions)

val_metrics = compute_pairwise_metrics(val_exact)
test_metrics = compute_pairwise_metrics(test_exact)

display(pd.DataFrame([
    {"split": "validation", **val_metrics},
    {"split": "test", **test_metrics},
]))


,split,precision,recall,f1,tp,fp,fn,num_gold_clusters,num_pred_clusters,num_mentions,num_topics
0,validation,0.914946,0.063249,0.118319,1108,103,16410,1747,3931,4873,100
1,test,0.917468,0.058075,0.109235,2290,206,37142,3711,8486,10423,200


In [3]:
# Inspect multi-mention clusters produced by Exact Matching on Test
cluster_sizes = test_exact.groupby("pred_cluster_id").size()
multi_cids = cluster_sizes[cluster_sizes >= 3].head(5).index
display(test_exact[test_exact["pred_cluster_id"].isin(multi_cids)][
    ["topic_id", "paper_id", "mention_text", "normalized_text", "gold_cluster_id", "pred_cluster_id"]
])


,topic_id,paper_id,mention_text,normalized_text,gold_cluster_id,pred_cluster_id
8307,1000,14266cb684c074468d3564a38b980799269c652b,cloud computing era,cloud computing era,test::t1000::c5,test::t1000::exact_1
8308,1000,37b9448ea40fabaef73d8b46871b2446e34fa4e0,cloud computing era,cloud computing era,test::t1000::c5,test::t1000::exact_1
8309,1000,5150d0b3181535455aadaa4c9c0e730253a61f1d,cloud computing era,cloud computing era,test::t1000::c5,test::t1000::exact_1
8314,1000,ce4821ea73d46933a9396374e4f487497f6a8c85,storage of data,storage of data,test::t1000::c4,test::t1000::exact_4
8315,1000,bce624a2ab2c0a341954f6fca696d6cb11772229,storage of data,storage of data,test::t1000::c4,test::t1000::exact_4
8316,1000,480d260325e5beb22cb370a67bb29dfb6928d60b,storage of data,storage of data,test::t1000::c4,test::t1000::exact_4
8346,1000,736e8c9282fb63b5b129fa39c72de0778cd47a83,cloud computing environment,cloud computing environment,test::t1000::c5,test::t1000::exact_27
8347,1000,8ca7edc5434e0e6d6533f3921223db89c279f435,cloud computing environment,cloud computing environment,test::t1000::c5,test::t1000::exact_27
8348,1000,7c681d82e583e5b69137689bc67e63891901c98c,cloud computing environment,cloud computing environment,test::t1000::c5,test::t1000::exact_27
8361,1000,233190c4a298843cf0fc5681ddfd2ee7b1c56c77,Cloud Computing domain,cloud computing domain,test::t1000::c5,test::t1000::exact_35


In [4]:
# Inspect common failure cases: same gold cluster split due to surface variation
from src.error_analysis import analyze_coreference_errors

fm_exact, mm_exact, exact_err_summary = analyze_coreference_errors(test_exact, max_examples_per_category=15)
print("Exact matching error totals on Test:", exact_err_summary)
print("Sample Missed Matches (FN) from Exact Matching:")
display(mm_exact[["topic_id", "mention_text_1", "mention_text_2", "likely_cause"]].head(10))
print("Sample False Merges (FP) from Exact Matching (homonyms):")
display(fm_exact[["topic_id", "mention_text_1", "mention_text_2", "likely_cause"]].head(10))


Exact matching error totals on Test: {'total_false_merges': 206, 'total_missed_matches': 37142, 'false_merge_causes': {'lexical_overlap_different_concept': 206}, 'missed_match_causes': {'partial_lexical_overlap_below_threshold': 30487, 'cross_paper_terminology_variation': 5474, 'acronym_vs_expanded_or_variant': 1174, 'different_surface_expression': 7}}
Sample Missed Matches (FN) from Exact Matching:


,topic_id,mention_text_1,mention_text_2,likely_cause
0,842,batch normalization,Batch Normalization configuration,partial_lexical_overlap_below_threshold
1,842,batch normalization,batch normalization process,partial_lexical_overlap_below_threshold
2,842,batch normalization,batch - normalization,partial_lexical_overlap_below_threshold
3,842,batch normalization,Batch Normalization Layers,partial_lexical_overlap_below_threshold
4,842,batch normalization,batch normalization layers,partial_lexical_overlap_below_threshold
5,842,Batch Normalization,Batch Normalization configuration,partial_lexical_overlap_below_threshold
6,842,Batch Normalization,batch normalization process,partial_lexical_overlap_below_threshold
7,842,Batch Normalization,batch - normalization,partial_lexical_overlap_below_threshold
8,842,Batch Normalization,Batch Normalization Layers,partial_lexical_overlap_below_threshold
9,842,Batch Normalization,batch normalization layers,partial_lexical_overlap_below_threshold


Sample False Merges (FP) from Exact Matching (homonyms):


,topic_id,mention_text_1,mention_text_2,likely_cause
0,842,channel normalization,channel normalization,lexical_overlap_different_concept
1,843,FPN ),FPN,lexical_overlap_different_concept
2,843,FPN,FPN,lexical_overlap_different_concept
3,843,FPN,FPN,lexical_overlap_different_concept
4,843,FPN,FPN ),lexical_overlap_different_concept
5,843,FPN,FPN,lexical_overlap_different_concept
6,857,randomized ensemble,randomized ensemble,lexical_overlap_different_concept
7,858,arrhythmia analysis,arrhythmia analysis,lexical_overlap_different_concept
8,858,heart rate variability measurements,heart rate variability measurements,lexical_overlap_different_concept
9,860,exponential formulation,exponential formulation,lexical_overlap_different_concept
